# S2 — Pandas I/O 與資料清理（解答版）

> **本篇為解答版 (Answer Key)**，建議先完成 `S2_pandas_io_cleaning.ipynb` 的練習後再對照。
> 
> ⚠️ **請先自己動手嘗試**，直接看答案學不到東西。卡關超過 15 分鐘再翻解答。

---

本解答版包含：
- 🟢 送分題：DataFrame 基本探索
- 🟡 核心題：`orders_raw.csv` 清理 + 回答 3 個問題
- 🔴 挑戰題：`clean_orders(path)` 可重用函式
- 💡 教學提示：常見陷阱與最佳實務

In [ ]:
import pandas as pd
import numpy as np

# 同 S2 使用相對路徑（notebook 位於 M2_Pandas_Basic/ 之下）
DATA = '../datasets/ecommerce/orders_raw.csv'

---
## 🟢 送分題 — DataFrame 基本探索

**題目**：建立下面這個 DataFrame，然後：
1. 用 `.head(2)` 看前兩列
2. 用 `.dtypes` 查看型別
3. 取出 `price` 這個 column

In [ ]:
# ✅ 解答
practice = pd.DataFrame({
    'item':  ['Apple', 'Banana', 'Cherry', 'Durian'],
    'price': [50, 20, 150, 300],
    'stock': [10, 50, 5, 2],
})

# (1) 前兩列 —— head(n) 是快速檢視資料的第一招
print('--- practice.head(2) ---')
print(practice.head(2))

# (2) 每欄型別 —— 拿到新資料一定要先看型別，很多 bug 都是型別不對造成的
print('\n--- practice.dtypes ---')
print(practice.dtypes)

# (3) 取出 price 欄 —— 單括號回傳 Series（一維），雙括號才是 DataFrame（二維）
print('\n--- practice["price"] (Series) ---')
print(practice['price'])
print('\n型別:', type(practice['price']).__name__)

💡 **教學提示（送分題）**：
- `df['col']` → 回傳 **Series**（一維）
- `df[['col']]` → 回傳 **DataFrame**（二維，只有一欄）
- 很多學生會困惑為什麼 `df[['a','b']]` 要兩層中括號 —— 因為外層是 `__getitem__`，內層是 list。

---
## 🟡 核心題 — 清理 orders_raw.csv 並回答問題

**題目**：重新讀一次 `orders_raw.csv`，完成清理並回答：
1. 清理後總共有多少筆訂單？
2. `amount` 的平均值是多少？
3. amount > 5000 的訂單有幾筆？

In [ ]:
# ✅ 解答 —— 逐步清理並回答三個問題

# Step 0: 重新讀檔（注意：每次從原始檔開始，避免污染）
df = pd.read_csv(DATA)
original_rows = df.shape[0]
print(f'原始資料: {original_rows} 筆')

# Step 1: 欄名標準化 —— 去空白 + 轉小寫（解決 'Order_ID ' 尾端空格問題）
df.columns = df.columns.str.strip().str.lower()

# Step 2: amount 字串轉數字
#   順序很重要：先 astype(str) → 再 replace → 最後 astype(float)
#   為什麼要先 astype(str)？因為有些列本來就是數字，.str 取值器只對字串有效
df['amount'] = (
    df['amount']
    .astype(str)
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .astype(float)
)

# Step 3: 日期欄轉型 —— errors='coerce' 讓無法解析的變 NaT，而不是整個報錯
df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')

# Step 4: 缺值處理
df = df.dropna(subset=['order_date'])                    # 沒日期無法做時序，直接丟
df['qty'] = df['qty'].fillna(df['qty'].median())         # qty 用中位數補（比 mean 穩健）

# Step 5: 去重
df = df.drop_duplicates()

# ---- 回答問題 ----
q1_total = len(df)
q2_mean = df['amount'].mean()
q3_big_orders = (df['amount'] > 5000).sum()

print(f'\n【Q1】清理後總筆數: {q1_total} 筆（原 {original_rows} → {q1_total}）')
print(f'【Q2】amount 平均值: {q2_mean:.2f}')
print(f'【Q3】amount > 5000 的訂單: {q3_big_orders} 筆')

💡 **教學提示（核心題）**：
- **忘記 `errors='coerce'`** 是最常見錯誤 → `to_datetime` 碰到爛資料會直接拋例外，整個 pipeline 停擺。加上 `coerce` 後無法解析的會變 `NaT`（Not a Time），再用 `dropna` 處理。
- **型別轉換順序**：如果 amount 欄混有 `int` 和 `str`，直接 `.str.replace` 會在 int 那格回傳 `NaN`。正解是先 `.astype(str)` 把全欄統一成字串再處理。
- **布林遮罩計數**：`(df['amount'] > 5000).sum()` 比 `len(df[df['amount']>5000])` 更簡潔，因為 `True=1, False=0`。

---
## 🔴 挑戰題 — 封裝成可重用函式 `clean_orders(path)`

**題目**：把清理邏輯封裝成函式，輸入髒檔案路徑、輸出乾淨 DataFrame，並印出清理摘要。

**為什麼要封裝？**
> 未來每週收到新的訂單檔，只要 `clean_orders('2026-04.csv')` 就清好了 —— 這就是 **DE 的 reusable pipeline**。一次寫好、千次呼叫。

In [ ]:
# ✅ 解答：可重用的訂單清理函式

def clean_orders(path: str) -> pd.DataFrame:
    """清理訂單原始檔，回傳乾淨 DataFrame。

    處理步驟：
      1. 欄名 strip + lower
      2. amount 移除 $ 與 , 後轉 float
      3. order_date 轉 datetime（errors='coerce'）
      4. 丟棄無日期的列
      5. qty 用中位數補值
      6. 移除完全重複的列

    參數
    ----
    path : str
        訂單 CSV 檔案路徑。

    回傳
    ----
    pd.DataFrame
        清理後的訂單 DataFrame。
    """
    # --- Extract：讀檔 ---
    df = pd.read_csv(path)
    n_before = len(df)  # 記錄原始筆數，用於最後印摘要

    # --- Transform Step 1: 欄名標準化 ---
    # strip() 去掉頭尾空白、lower() 全轉小寫 → 避免 'Order_ID ' 這類陷阱
    df.columns = df.columns.str.strip().str.lower()

    # --- Transform Step 2: amount 字串→數字 ---
    # 先 astype(str) 統一型別，再剝除 $ 和 ,，最後 cast 成 float
    df['amount'] = (
        df['amount']
        .astype(str)
        .str.replace('$', '', regex=False)
        .str.replace(',', '', regex=False)
        .astype(float)
    )

    # --- Transform Step 3: 日期轉型（errors='coerce' 是關鍵！）---
    df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')

    # --- Transform Step 4: 缺值處理 ---
    df = df.dropna(subset=['order_date'])              # 無日期 → 丟
    df['qty'] = df['qty'].fillna(df['qty'].median())   # qty 缺 → 中位數補

    # --- Transform Step 5: 去重 ---
    df = df.drop_duplicates()

    # --- 印出 1 行清理摘要 ---
    n_after = len(df)
    print(f'清理完成：原 {n_before} 筆 → {n_after} 筆')

    return df


# 測試函式
clean = clean_orders(DATA)
print('\n--- 清理後 dtypes ---')
print(clean.dtypes)
print('\n--- 清理後 head(3) ---')
print(clean.head(3))

💡 **教學提示（挑戰題）—— 封裝函式的常見陷阱**：

1. **不要改動傳入的 DataFrame**：函式內操作的是 `pd.read_csv` 回傳的新物件，不會影響外部變數（符合不可變性原則）。
2. **`errors='coerce'` 一定要加**：沒加的話，只要一列日期格式錯誤就整個 pipeline 炸掉。
3. **`.str.replace` vs 數字轉換順序**：若先 `astype(float)` 再 `.str.replace` 會報錯，因為 float 沒有 `.str` 取值器。記住順序：**先清字串，再轉型別**。
4. **回傳值**：函式**一定要 `return df`**，不然呼叫端拿到 `None`。這是新手最常犯的錯誤之一。
5. **`drop_duplicates` 的時機**：放在最後，因為前面的清理可能讓原本看似不同的列變成相同（例如日期格式標準化後）。
6. **用 median 而非 mean 補 qty**：中位數對極端值更穩健，不會被 1 筆異常大單拉偏。

### 🚀 進階延伸（給學有餘力的同學）
- 加上 `logging` 取代 `print`，方便在 production 追蹤
- 加上 schema 驗證（例如用 `pandera`）確保輸出欄位和型別都正確
- 把函式寫成 class `OrderCleaner`，支援更多可配置參數（例如自訂缺值策略）

---
## 🎯 解答版小結

| 題目 | 關鍵技巧 | 常見錯誤 |
|---|---|---|
| 🟢 送分題 | `head` / `dtypes` / 欄位選取 | 混淆 `df['x']` 與 `df[['x']]` |
| 🟡 核心題 | strip → 字串清洗 → 日期轉型 → dropna | 忘記 `errors='coerce'` |
| 🔴 挑戰題 | 封裝為可重用函式 | 忘記 `return df`、型別順序錯誤 |

**核心心法**：拿到髒資料 → `head/info/isna` 掃一遍 → 逐欄處理 → 封裝成函式 → 未來只呼叫一行。

下一節 **S3** 會用這份清乾淨的資料做**三表 join** 與**分組聚合**，正式進入商業洞察階段。